# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing specific dataset entities by their `@id` fields to ensure reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Show general metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, their `@id`, and associated fields (columns). This helps select the record sets and fields for loading and analysis.

**Note:** All entities are referenced by their `@id` as required by the dataset schema.

In [ ]:
# List all record sets in the dataset by their @id and description.
print("Record sets available in the dataset:")
for rs in metadata.record_sets:
    print(f"  - @id: {rs.id}     name: {rs.name if hasattr(rs, 'name') else 'N/A'}")

# For demonstration, display all fields (columns) in each record set.
for rs in metadata.record_sets:
    print(f"\nRecord set @id: {rs.id}  (name: {rs.name if hasattr(rs, 'name') else 'N/A'})")
    print("Fields / columns:")
    if hasattr(rs, 'columns') and rs.columns:
        for col in rs.columns:
            print(f"  - @id: {col.id}    name: {col.name if hasattr(col, 'name') else 'N/A'}    dataType: {col.data_type if hasattr(col, 'data_type') else 'N/A'}")
    else:
        print("  (No columns found in this record set.)")

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis.

Each table/record set is referenced by its `@id`, and columns/fields by their own `@id`.

In [ ]:
# Collect all record set @ids for extraction
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}
print(f"Extracting the following record sets: {record_set_ids}")

# Extract each record set into a DataFrame
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set {rs_id}: {df.shape} (rows, columns)")
    else:
        print(f"No records found for record set {rs_id}")

# For further demonstration, pick the first record set (if any loaded):
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Columns available in record set {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, grouping, etc. All actions reference columns by their `@id`, and record set by its `@id`.

**Please adjust the example below to use the actual desired fields.**

In [ ]:
import numpy as np

# Use the main DataFrame and pick a numeric field
record_set_id = main_record_set_id  # the first available record set
df = dataframes[record_set_id]

# Identify a numeric column by attempting to find a float/int column in the schema
record_set = next(rs for rs in metadata.record_sets if rs.id == record_set_id)
numeric_col_id = None
for col in getattr(record_set, 'columns', []):
    dt = getattr(col, 'data_type', None)
    if dt in ('Float', 'Integer', 'Number') or (dt and dt.lower() in ('float', 'integer', 'number')):
        if col.id in df.columns:
            numeric_col_id = col.id
            print(f"Selected numeric field for EDA: {numeric_col_id} (type: {dt})")
            break

if numeric_col_id is None:
    # Fallback: use the first numeric column in the DataFrame
    print("Could not identify numeric field via schema; checking DataFrame's dtypes.")
    for col_name in df.columns:
        if np.issubdtype(df[col_name].dtype, np.number):
            numeric_col_id = col_name
            print(f"Selected numeric field from DataFrame: {numeric_col_id}")
            break

if numeric_col_id:
    numeric_series = pd.to_numeric(df[numeric_col_id], errors='coerce')
    threshold = numeric_series.mean()  # Use mean as threshold for demo
    filtered_df = df[numeric_series > threshold].copy()
    print(f"Filtered records with {numeric_col_id} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_col_id}_normalized"] = (numeric_series[numeric_series > threshold] - numeric_series.mean()) / numeric_series.std()
    print(f"Normalized {numeric_col_id} for filtered records:")
    display(filtered_df[[numeric_col_id, f"{numeric_col_id}_normalized"]].head())

    # Attempt group by a string/categorical field
    group_field_id = None
    for col in getattr(record_set, 'columns', []):
        if getattr(col, 'data_type', '').lower() in ('text', 'string', 'categorical', 'str', 'category') and col.id in df.columns:
            group_field_id = col.id
            break
    # If none found, just use the second column as example
    if not group_field_id and len(df.columns) > 1:
        group_field_id = df.columns[1]

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_col_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} and averaged {numeric_col_id}:")
        display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships in the dataset. Plots reference column(s) by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_col_id:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_col_id].apply(pd.to_numeric, errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_col_id}")
    plt.xlabel(numeric_col_id)
    plt.ylabel("Count")
    plt.show()
    
    # If a group field was determined, do boxplot/grouped bar plot
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(y=numeric_col_id, x=group_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_col_id} grouped by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset _Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells_ using the `mlcroissant` library. By leveraging Croissant's rich metadata, we identified record sets and fields by their `@id`, extracted tabular data for analysis, and performed several standard EDA and visualization routines.

**Key findings:**
- The dataset provides extensive metadata and structured tables referencing proteomic measurements.
- Numeric fields can be identified programmatically from schema descriptors and used for filtering, normalization, and visualization.

**Next steps:**
You can adapt this notebook to focus on specific biological hypotheses or statistical questions, or extend with additional analyses (e.g., protein clustering, outlier analysis, data integration). For detailed field definitions, please refer to the Croissant schema documentation linked above.